2# Lab 2: Annotation with SynDisco

**Why this lab.** In Assignment 1 you compare yourself with a model that annotates like a person. This lab is **a way to approach it**: you walk through the same steps once, on just **5 texts** and with the code already written, so that when you do the assignment you know what each step is and how SynDisco fits in.

**The steps** (the same as in the assignment, in small):

1. you **rate** the texts yourself: your ratings are the reference;
2. a language model writes a short **conversation** about each text;
3. an **annotator** (a model with no persona, the *baseline*) rates the same texts, first alone and then after reading the conversation;
4. you look at **one table** with the three sets of ratings;
5. finally, you give the annotator a **persona** (a woman, a man) and compare again.

There is no analysis here. **Run the cells in order and read the explanation above each one**: the only thing you do by hand is the rating in step 1.

**Before you start:** on Colab, choose *Runtime → Change runtime type → T4 GPU*.

### Words used in this lab

| Word | Meaning |
|---|---|
| **annotation** | reading a text and giving it a label (here: how toxic it is) |
| **rating / label** | the **rating** is the 1-5 number. The **label** is `civil` (rating 1-2) or `toxic` (rating 3-5): a rule we apply, not something you choose |
| **Actor** | SynDisco's object for "someone who speaks": a model plus a persona, a context and instructions |
| **persona** | who the actor is: a small dictionary of attributes, such as `{"age": 38, "job": "nurse"}` |
| **context** | the situation the actor is in (what it knows before it speaks) |
| **instructions** | what the actor must do |
| **Logs** | the list of what was said, one entry per comment |
| **baseline** | an annotator with **no** persona: what the model does by default |

## 0. Setup

The first cell installs **SynDisco**, the library that runs the conversations and the annotators. The second one imports what we need and checks that you have a GPU: without it the model is too slow.

In [ ]:
!pip install -q syndisco

In [ ]:
import json                 # to save the conversations
import random               # to choose random participants
import re                   # to find the rating in the model's answer

import pandas as pd         # for the tables
import torch                # the library that runs the model
import syndisco             # the library of this lab

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

## 1. The texts, and your ratings

Below are five comments from news-site discussions. They go from civil to hostile, and they may contain harsh language.

Your job: rate each one from **1 (fully civil) to 5 (very toxic)**. Careful readers can disagree about this, and nobody is wrong: that is why we use your ratings as a *reference point* and not as "the truth".

Rate **alone and before you look at any model's answer**, so that the model cannot influence you.

In [ ]:
# the five texts to annotate: a dictionary {id: text}
texts = {
    1: "Find this hard to believe as most users that I know are dumping Rogers due to higher prices and poor customer service.",
    2: "Yup my mother at 88 and going strong has NEVER drank bottled water. Better warn her now before it too late. What an idiotic remark...has to go down as one of the worst comments ever on this site.",
    3: "Interesting how you're in favor of Trump saying what he thinks, but not in favor of quiet civil disobedience. Also, if we're going to complain about spoiled millionaires.....Oh, you know what I'm about to say about hypocrisy.",
    4: "Disagreement on Bill Morneaus conduct? Well, fair enough, that's democracy. Blaming Harper for all this mess? Now that's just plain dumb.",
    5: "The real reason that Trump-lite Conservatives want to destroy public education is to generate even more dumb supporters. Please stop the World, I want to get off.",
}
for i, t in texts.items():
    print(f"[{i}] {t}\n")

**Run the next cell and type your rating** for each text in the box that appears (then press Enter). If you type something that is not a whole number from 1 to 5, it asks again.

In [ ]:
my_ratings = {}                                   # will hold {id: your rating}

for i, t in texts.items():                        # one text at a time
    print(f"\n[{i}] {t}")

    while True:                                   # repeat the question until we get a valid answer
        answer = input("Your rating, 1 (fully civil) to 5 (very toxic): ").strip()   # what you typed
        if answer in {"1", "2", "3", "4", "5"}:   # valid: it is exactly one of these five
            break                                 # leave the "while" and go on
        print("Please type a whole number from 1 to 5.")   # not valid: say so, and ask again

    my_ratings[i] = int(answer)                   # keep it, as a number ("4" -> 4)

# put the ratings in a table and add the label with our rule: 1-2 civil, 3-5 toxic
mine = pd.DataFrame({"id": list(texts), "text": list(texts.values()), "rating": list(my_ratings.values())})
mine["label"] = mine["rating"].apply(lambda r: "civil" if r <= 2 else "toxic")
mine.to_csv("my_ratings.csv", index=False)        # saved in the Colab files (left panel)
mine[["id", "rating", "label"]]

## 2. The model

A *language model* writes text one word at a time. SynDisco does not contain a model: it **wraps** one from Hugging Face (`TransformersModel`) and gives it a prompt.

We use a small one, **Qwen2.5-3B-Instruct**, in a compressed form (**4-bit**, about 2 GB), so that it fits the free T4. It will be used twice: to write the conversations and to be the annotator.

Two settings to know:
- `max_out_tokens`: the longest answer it may write. A reply needs about 80; a rating needs only a few (we lower it later).
- `repetition_penalty`: makes the model less likely to repeat itself, which small models tend to do.

The model is *deterministic*: the same input always gives the same output. Variety will come from the random participants, not from the model.

Loading takes a minute or two (it downloads the model the first time).

In [ ]:
model = syndisco.TransformersModel(
    model_path="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   # which model to download
    name="qwen",                                         # a name for the logs
    max_out_tokens=80,                                   # longest answer: enough for a short reply
    generation_kwargs={"repetition_penalty": 1.1},       # avoids repeating the previous comment
)

## 3. One conversation per text

For each text we simulate a short discussion: **3 random participants** reply to it in turn. The text opens the thread, written by a made-up "OriginalPoster", and the participants write **4 replies**.

Each participant is an **`Actor`**, built from three things the model is told:
- a **persona**: who it is (a dictionary: name, age, job, personality). We use it only to make the replies different from each other;
- a **context**: the situation ("you are a commenter on a news website");
- **instructions**: what to do ("write one short reply, in the first person").

SynDisco does the rest: the **turn manager** decides who speaks next (`RandomTurnManager`: a random participant, never the same one twice in a row), and the **`Discussion`** keeps calling the next speaker until it has 4 replies.

The participants are chosen with `random.Random(0)`: the same participants every time you run it.

In [ ]:
# a small pool of made-up people: each is just a dictionary of attributes
PERSONAS = [
    {"username": "Maria_K", "age": 38, "job": "nurse", "personality": ["patient", "direct"]},
    {"username": "Giannis", "age": 21, "job": "game developer", "personality": ["sarcastic", "curious"]},
    {"username": "Kimya", "age": 67, "job": "retired teacher", "personality": ["strict", "grumpy"]},
    {"username": "Tom_77", "age": 45, "job": "truck driver", "personality": ["blunt", "funny"]},
    {"username": "Sofia", "age": 29, "job": "journalist", "personality": ["skeptical", "polite"]},
]

# the same context and instructions for every participant
CONTEXT = (
    "You are a commenter in a discussion thread on a news website. This is a role-play for a research "
    "study on online conversations: write the kind of reply that real commenters write, whether you agree or not."
)
INSTRUCTIONS = (
    "Write ONE short reply (one or two sentences) to the last comments, in the first person, "
    "as a real commenter would. React to what was said. Do not explain, do not refuse, "
    "do not mention that you are an AI, and never answer with an apology or a refusal."
)

In [ ]:
 rng = random.Random(0)           # change the 0 to get other participants
conversations = {}               # will hold {text id: the conversation}

for i, t in texts.items():
    # 1. choose 3 of the 5 people and turn each one into an Actor (a model + persona + context + instructions)
    users = [
        syndisco.Actor(model=model, persona=p, context=CONTEXT, instructions=INSTRUCTIONS, name=p["username"])
        for p in rng.sample(PERSONAS, 3)
    ]
    # 2. set up the discussion
    discussion = syndisco.Discussion(
        next_turn_manager=syndisco.RandomTurnManager(users),  # who speaks next
        users=users,
        conv_len=4,                          # number of replies the model writes
        seed_opinions=[t],                   # the text itself opens the thread...
        seed_opinion_usernames=["OriginalPoster"],            # ...as if written by this user
    )
    # 3. run it and keep the log (the list of comments)
    discussion.begin(verbose=False)
    conversations[i] = discussion.get_logs()
print("done")

**Read the conversations.** Each line is `who: what they said`; the first one is the text itself.

Do they make sense? Small models often make the same mistakes: a reply is **cut off**, **repeats** the previous one, or **refuses** ("I'm sorry, but I can't assist..."). If one conversation is bad, you can change the `0` in `random.Random(0)` above and run that cell again to get other participants.

In [ ]:
def as_text(logs):
    # one line per comment: "name: text"
    return "\n".join(f'{e["name"]}: {e["text"]}' for e in logs)

for i, logs in conversations.items():
    print(f"--- conversation {i} ---")
    print(as_text(logs), "\n")

Now we **save** the conversations to a file, one line per conversation (`.jsonl`). Assignment 1 asks you to hand in the same file.

In [ ]:
with open("generated.jsonl", "w", encoding="utf-8") as f:
    for i, logs in conversations.items():
        entry = {"id": i, "conversation": [{"name": e["name"], "text": e["text"]} for e in logs]}
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
print("saved generated.jsonl")

## 4. The annotator

An annotator is also an **`Actor`**, with `is_annotator=True`. It is built from the same three things:

| | In this lab |
|---|---|
| **persona** | none: this is the **baseline** (in Assignment 1 you will give personas to the annotators) |
| **instructions** | the same for everyone: rate from 1 to 5 and answer only with the number |
| **context** | **the only thing that changes**: empty in one run, the conversation in the other |

So we run the annotator twice on each text:
- **without context**: it only sees the text;
- **with context**: it first reads the conversation we generated, and then rates **the text** (not the conversation).

How it works: SynDisco gives the annotator a *log*, and the annotator rates **every entry** of it. We put **one entry** in the log, the text, so the rating is the rating of the text. The conversation goes in the `context` instead, so that it is read but not rated.

A rating needs only a few tokens, so we lower `max_out_tokens`. And since models do not always obey "reply only with a number", we **look for the first digit from 1 to 5** in the answer, and keep `None` if there is none.

In [ ]:
RATING_INSTRUCTIONS = (
    "How toxic is the comment? Answer with one number from 1 (fully civil) to 5 (very toxic). "
    "Reply only with the number."
)
model.max_out_tokens = 5     # a rating needs only a few tokens (the replies needed 80)


def rate(text, conversation=None, persona=None):
    # the context: the same sentence always; the conversation is added only in the "with context" run
    context = "You are annotating comments from an online discussion."
    if conversation:
        context += " Before you rate, read the conversation that people had about the comment:\n" + conversation

    # the annotator: persona=None is the baseline (no identity); same instructions for everyone
    name = "baseline" if persona is None else ", ".join(persona.values())     # only a label for the logs
    annotator = syndisco.Actor(
        model=model, persona=persona, context=context, instructions=RATING_INSTRUCTIONS,
        is_annotator=True, name=name,
    )

    # the log it has to annotate: one entry, the text
    logs = syndisco.Logs() # Creates an empty log
    logs.append("comment", text)

    # run the annotation and read its answer
    job = syndisco.Annotation(annotator=annotator, discussion_logs=logs)
    job.begin(verbose=False)
    answer = job.get_logs()[0]["text"]

    found = re.search(r"[1-5]", answer)             # the first digit from 1 to 5
    return int(found.group()) if found else None    # None = the model did not answer with a rating


no_context = {i: rate(t) for i, t in texts.items()}                                  # only the text
with_context = {i: rate(t, as_text(conversations[i])) for i, t in texts.items()}     # text + conversation
print("done")

## 5. The table

One row per text, with the three ratings side by side:

- **you**: your rating from step 1;
- **model, no context**: the baseline annotator, which saw only the text;
- **model, with context**: the baseline annotator, after reading the conversation.

`None` means that the model did not answer with a number from 1 to 5.

In [ ]:
table = pd.DataFrame({
    "id": list(texts),
    "text": [t[:60] + "..." for t in texts.values()],      # the first 60 characters, to keep the table short
    "you": list(my_ratings.values()),
    "model, no context": list(no_context.values()),
    "model, with context": list(with_context.values()),
})
table.to_csv("lab_results.csv", index=False)
table

**Look at the table.** Where do you and the model disagree? Did reading the conversation change what the model said, and in which direction? There is no right answer: it is just 5 texts.

## 6. A persona: one dimension, two values

So far the annotator had no persona. Now we give it one. A persona is **who the annotator is**: a dictionary of attributes that goes into its prompt. We use a single dimension, **gender**, with two values, so there are two annotators:

- `{"gender": "woman"}`
- `{"gender": "man"}`

We do **not** tell them how to rate. The instructions are exactly the same as for the baseline, and the model decides how a woman or a man would rate. The only thing that changes is the persona. Does it change the ratings?

To keep it short we run them **without context** (only the text). Adding the conversation works exactly as in step 4.

In [ ]:
personas = {
    "woman": {"gender": "woman"},
    "man": {"gender": "man"},
}

# the same function as before, now giving the annotator a persona
by_persona = {
    name: {i: rate(t, persona=p) for i, t in texts.items()}
    for name, p in personas.items()
}
print("done")

In [ ]:
# the same table, with the baseline (no context) and the two personas next to you
personas_table = table[["id", "text", "you", "model, no context"]].rename(columns={"model, no context": "baseline"})
for name in personas:
    personas_table[name] = list(by_persona[name].values())
personas_table.to_csv("lab_results_personas.csv", index=False)
personas_table

**Look at it.** Does the woman annotator rate like the man annotator? Do they rate like the baseline? Which one is closest to **you**? Compare also with the person next to you: that is the same question, a person against a person.

A small model may ignore the persona and give the same ratings to everyone. If so, that is a result too, not a mistake.

That is the question of **Assignment 1**, which asks it on **20 texts**, with a **grid of personas** that you design, and measures the agreement properly.

### From this lab to Assignment 1

| In this lab | In Assignment 1 |
|---|---|
| you rate 5 texts in the notebook | you rate 20 texts in a web page (parts A and B) |
| 3 random participants, 4 replies | conversations that you design and check (part C) |
| the baseline and 2 personas (gender) | a grid of persona annotators that you design, plus the baseline (parts D and E) |
| you look at two tables | you measure agreement and interpret it (part F) |

Files created in this lab (see the folder icon on the left): `my_ratings.csv`, `generated.jsonl`, `lab_results.csv`, `lab_results_personas.csv`.